# Inverse-Probability-Weighted Logit & Survival

Converted from `Code/r_02_AIPW.R` — R-kernel Jupyter port. Outputs tables to `Output/Tables/` and figures to `Output/Images/Graphs/`. Run the setup cell first, then sections in order.

In [1]:
# AIPW.R — Inverse-Probability-Weighted (IPW/CBPS) logit and survival models
# Sections:
#   1. Main specifications: total and split (small/large) monastic land — 6 specs
#   2. OwnOther specification: on-site vs off-site land — 3 specs
# Each section produces stargazer tables and coefficient plots.

pacman::p_load(
  sf, tidyverse, stargazer, sp, dplyr,
  cem, MatchIt, WeightIt, marginaleffects, ipw,
  survey, optmatch, conflicted, cobalt, twang,
  survival, ggplot2, broom, jsonlite
)
conflict_prefer("filter", "dplyr")
conflict_prefer("select", "dplyr")

PROJECT_ROOT <- tryCatch(
  normalizePath(file.path(dirname(rstudioapi::getActiveDocumentContext()$path), "..")),
  error = function(e) {
    cwd <- normalizePath(getwd())
    if (basename(cwd) == "Code") dirname(cwd) else cwd
  }
)
setwd(PROJECT_ROOT)
# Load pretty dictionary for labels
pretty_dict <- fromJSON("Code/pretty_dict.json")

pdf <- read_sf(dsn = "Data/Processed/northParishFlows.shp")

# Replace NAs in terrainTyp with 'Other'
pdf$terrainTyp <- ifelse(is.na(pdf$terrainTyp), "Other", pdf$terrainTyp)
pdf$uplands    <- ifelse(pdf$terrainTyp == "Uplands",  1, 0)
pdf$lowlands   <- ifelse(pdf$terrainTyp == "Lowlands", 1, 0)
pdf$otherlands <- ifelse(pdf$terrainTyp == "Other",    1, 0)

rdf <- data.frame(pdf)
day <- 40
rdf$day <- replace(rdf$day, rdf$day < 1, day)
rdf$day <- ifelse(is.na(rdf$day), day, rdf$day)
rdf$primary_day <- rdf$day * rdf$primary
rdf$primary_day <- replace(rdf$primary_day, rdf$primary_day < 1, day)
rdf$survival <- rdf$day - rdf$news_day
rdf$primary_survival <- rdf$primary_day - rdf$news_day
rdf$primary_survival <- ifelse(is.na(rdf$primary_survival), day, rdf$primary_survival)

# Convert seats to binary (1 if seats > 1, 0 otherwise)
rdf$seats <- ifelse(rdf$seats > 1, 1, rdf$seats)

# Standardize and center continuous variables.
# Binary dummies (smHouse, bigHouse, mg_fsnub, mg_court, friary) are NOT standardized.
for (v in c(
  # Total monastic land (3 normalizations)
  "llandOwned", "llo_sk", "llo_arak",
  # Small/large split land (3 normalizations)
  "lsmLand",   "lbigLand",
  "lsm_sk",    "lbg_sk",
  "lsm_arak",  "lbg_arak",
  # Off-site/on-site split land (3 normalizations)
  "lotherLand", "lownLand",
  "loth_sk",    "lown_sk",
  "loth_arak",  "lown_arak",
  # Tithes, alms, net income (3 normalizations each)
  "ltitheOutT", "lti_sk", "lti_arak",
  "lalmsInTot", "lal_sk", "lal_arak",
                "lni_sk", "lni_arak",
  # Controls (continuous)
  "lLStax_pc", "lpopC", "distScot", "area", "mean_slope",
  "wet_1535", "wet_1536"
)) {
  rdf[[v]] <- scale(rdf[[v]], center = TRUE, scale = TRUE)[, 1]
}

# --- Shared "after monastic" covariates common to every main spec ---------
# smHouse / bigHouse / mg_fsnub / mg_court: binary 20km proximity dummies — not standardized.
main_shared_after <- c("smHouse", "bigHouse", "friary", "mg_fsnub", "mg_court")

# --- Shared geographic / socioeconomic controls ---------------------------
covar_rhs <- c(
  "lLStax_pc", "lpopC", "distScot", "area",
  "uplands", "lowlands", "mean_slope", "wet_1535", "wet_1536"
)

# --- Six main specifications (total and split, 3 normalizations each) -----
# treatment = main explanatory variable (large house or total land)
# mon_covars = complementary monastic variables in the outcome model
main_specs <- list(
  total_raw = list(
    treat      = "llandOwned",
    mon_covars = c("ltitheOutT", "lalmsInTot"),
    suffix     = "_total_raw"
  ),
  total_sk = list(
    treat      = "llo_sk",
    mon_covars = c("lti_sk", "lal_sk"),
    suffix     = "_total_sk"
  ),
  total_arak = list(
    treat      = "llo_arak",
    mon_covars = c("lti_arak", "lal_arak", "lni_arak"),
    suffix     = "_total_arak"
  ),
  split_raw = list(
    treat      = "lbigLand",
    mon_covars = c("lsmLand", "ltitheOutT", "lalmsInTot"),
    suffix     = "_split_raw"
  ),
  split_sk = list(
    treat      = "lbg_sk",
    mon_covars = c("lsm_sk", "lti_sk", "lal_sk"),
    suffix     = "_split_sk"
  ),
  split_arak = list(
    treat      = "lbg_arak",
    mon_covars = c("lsm_arak", "lti_arak", "lal_arak", "lni_arak"),
    suffix     = "_split_arak"
  )
)


[conflicted] Will prefer dplyr::filter over any other package.


[conflicted] Will prefer dplyr::select over any other package.


## Helper: coefficient extraction and plotting

In [2]:
extract_coefs_svyglm <- function(model, var_name) {
  coef_summary <- summary(model)$coefficients
  coef_val <- coef_summary[var_name, "Estimate"]
  se_val   <- coef_summary[var_name, "Std. Error"]
  z_crit   <- qnorm(0.95)  # 90% CI
  ci_lower <- coef_val - z_crit * se_val
  ci_upper <- coef_val + z_crit * se_val
  z_stat   <- coef_val / se_val
  p_val    <- 2 * pnorm(abs(z_stat), lower.tail = FALSE)
  data.frame(variable = var_name, coefficient = coef_val, se = se_val,
             ci_lower = ci_lower, ci_upper = ci_upper, p_value = p_val)
}

extract_coefs_coxph <- function(model, var_name) {
  coef_summary <- summary(model)$coefficients
  coef_val <- coef_summary[var_name, "coef"]
  se_val   <- coef_summary[var_name, "se(coef)"]
  z_crit   <- qnorm(0.95)
  ci_lower <- coef_val - z_crit * se_val
  ci_upper <- coef_val + z_crit * se_val
  z_stat   <- coef_val / se_val
  p_val    <- 2 * pnorm(abs(z_stat), lower.tail = FALSE)
  data.frame(variable = var_name, coefficient = coef_val, se = se_val,
             ci_lower = ci_lower, ci_upper = ci_upper, p_value = p_val)
}

make_coef_df_ipw <- function(model, vars, extract_fn) {
  coefs <- bind_rows(lapply(vars, function(v) extract_fn(model, v)))
  coefs$significant <- ifelse(is.na(coefs$p_value), FALSE, coefs$p_value < 0.10)
  coefs$order       <- match(coefs$variable, vars)
  coefs
}

make_ipw_plot <- function(coef_df, var_labels, x_label = "Coefficient (Log Odds)") {
  # Look up pretty labels; unlist first so missing keys yield NA (preserving length)
  labels_vec <- unlist(var_labels)
  coef_df$variable_label <- unname(labels_vec[coef_df$variable])
  # Pre-compute ordered factor outside aes() to avoid IRkernel tidy-eval quirk
  lvl_order <- order(-coef_df$order)
  coef_df$variable_label <- factor(coef_df$variable_label,
                                    levels = coef_df$variable_label[lvl_order])
  ggplot(coef_df, aes(x = coefficient, y = variable_label)) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "gray50") +
    geom_errorbar(aes(xmin = ci_lower, xmax = ci_upper),
                  width = 0.2, color = "gray30", orientation = "y") +
    geom_point(aes(color = significant), size = 3) +
    scale_color_manual(
      values = c("FALSE" = "gray60", "TRUE" = "#0072B2"),
      labels = c("FALSE" = "Not Significant", "TRUE" = "p < 0.10")
    ) +
    labs(x = x_label, y = "", color = "Significance") +
    theme_minimal() +
    theme(
      axis.text.x  = element_text(size = 16),
      axis.text.y  = element_text(size = 16),
      axis.title.x = element_text(size = 16),
      legend.text  = element_text(size = 15),
      legend.title = element_text(size = 15),
      legend.position = "bottom"
    )
}


In [3]:
# --- Weighted Conley (spatial HAC) sandwich for IPW svyglm logit ----------
# conleyreg does not accept observation weights, so we implement the weighted
# spatial-HAC sandwich directly (same as Section 3, moved here for reuse).
# Returns a named numeric vector of SEs aligned to coef(svy_mod).

parish_xy_ipw_main <- sf::st_coordinates(sf::st_centroid(sf::st_geometry(pdf)))
rdf$.cx <- parish_xy_ipw_main[, 1]
rdf$.cy <- parish_xy_ipw_main[, 2]

weighted_conley_se <- function(svy_mod, wts_full, cutoff_km = 100,
                               kernel = "bartlett") {
  mf <- model.frame(svy_mod)
  X  <- model.matrix(svy_mod)
  y  <- model.response(mf)
  mu <- as.numeric(fitted(svy_mod))
  used_rows <- as.integer(rownames(mf))
  w  <- wts_full[used_rows]
  cx <- rdf$.cx[used_rows]; cy <- rdf$.cy[used_rows]

  fam <- svy_mod$family$family
  if (grepl("binomial", fam)) {
    Avar <- mu * (1 - mu)
  } else if (grepl("poisson", fam)) {
    Avar <- mu
  } else {
    Avar <- rep(1, length(mu))
  }

  WX_bread <- X * (w * Avar)
  bread    <- solve(crossprod(X, WX_bread))
  u  <- as.numeric(w * (y - mu))
  Xu <- X * u

  d_km <- as.matrix(stats::dist(cbind(cx, cy))) / 1000
  K    <- pmax(1 - d_km / cutoff_km, 0)   # Bartlett kernel

  meat <- crossprod(Xu, K %*% Xu)
  V    <- bread %*% meat %*% bread
  setNames(sqrt(diag(V)), colnames(X))
}

## Section 1: Main specification (sm/bg monastery land per arable km²)

In [4]:
# Fit CBPS-weighted logit and Cox PH models for every main spec (6 total).
# Results stored in main_results for use in Conley / Moran's I sections.
main_results <- list()

for (spec_name in names(main_specs)) {
  spec   <- main_specs[[spec_name]]
  sfx    <- spec$suffix
  tr     <- spec$treat
  monc   <- spec$mon_covars
  covars <- c(monc, main_shared_after, covar_rhs)
  rhs    <- paste(c(tr, covars), collapse = " + ")

  cat(sprintf("\n===== IPW spec [%s] — treatment %s =====\n", spec_name, tr))

  wt <- weightit(
    as.formula(paste(tr, "~", paste(covars, collapse = " + "))),
    data = rdf, method = "cbps", over = FALSE
  )
  wts    <- wt$weights
  design <- svydesign(~1, weights = wts, data = rdf)

  wlm_primary <- svyglm(as.formula(paste("primary ~", rhs)),
                        data = rdf, weights = wts,
                        design = design, family = quasibinomial())
  wlm_muster  <- svyglm(as.formula(paste("muster ~",  rhs)),
                        data = rdf, weights = wts,
                        design = design, family = quasibinomial())
  wlm_seats   <- svyglm(as.formula(paste("seats ~",   rhs)),
                        data = rdf, weights = wts,
                        design = design, family = quasibinomial())
  wsurv <- coxph(as.formula(paste("Surv(primary_survival, primary) ~", rhs)),
                 data = rdf, weights = wts, robust = TRUE)

  main_results[[spec_name]] <- list(
    muster = wlm_muster, primary = wlm_primary,
    seats  = wlm_seats,  surv    = wsurv,
    wts = wts, tr = tr, covars = covars, monc = monc
  )

  # Back-compat aliases: split_arak is the primary per-arable-km² specification
  if (spec_name == "split_arak") {
    main_treat       <<- tr
    main_covars      <<- covars
    main_formula_rhs <<- rhs
    wt_main            <<- wt
    weights_main       <<- wts
    design_main        <<- design
    wlm_primary_main   <<- wlm_primary
    wlm_muster_main    <<- wlm_muster
    wlm_seats_main     <<- wlm_seats
    wsurv_main         <<- wsurv
  }

  print(summary(wlm_primary))
  print(summary(wsurv))

  # Covariate label order: treatment → complementary monastic vars → dummies → controls
  ipw_cov_order  <- c(tr, monc, main_shared_after,
                      "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  ipw_cov_labels <- unlist(pretty_dict[ipw_cov_order])

  # Conley SEs (100km Bartlett) for the three svyglm logit columns.
  # coxph with robust = TRUE already carries Lin-Wei robust SEs.
  se_muster  <- weighted_conley_se(wlm_muster,  wts)
  se_primary <- weighted_conley_se(wlm_primary, wts)
  se_seats   <- weighted_conley_se(wlm_seats,   wts)

  stargazer(wlm_muster, wlm_primary, wlm_seats, wsurv,
    type = "latex",
    title = paste0("IPW Logit and Cox PH Models — Muster, Primary, Seats [", spec_name, "]"),
    label = paste0("tab:ipw", sfx),
    align = TRUE,
    table.placement = "H",
    column.labels = c("Muster", "Primary", "Seats", "Cox PH"),
    se = list(se_muster, se_primary, se_seats, NULL),
    order = paste0("^", ipw_cov_order, "$"),
    covariate.labels = ipw_cov_labels,
    omit = c("Constant", "uplands", "lowlands", "area", "mean_slope"),
    add.lines = list(
      c("Geographic Controls",  "Y", "Y", "Y", "Y"),
      c("CBPS weights",         "Y", "Y", "Y", "Y"),
      c("Conley SEs (100 km)",  "Y", "Y", "Y", "—"),
      c("Robust SEs (Lin-Wei)", "—", "—", "—", "Y")
    ),
    column.sep.width = ".5pt",
    omit.stat = c("aic", "lr", "wald", "logrank"),
    out = paste0("Output/Tables/IPW", sfx, ".tex")
  )

  vars_to_plot <- c(tr, monc, main_shared_after,
                    "lLStax_pc", "wet_1535", "wet_1536", "lpopC")

  ggsave(paste0("Output/Images/Graphs/ipw_logit_primary_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_primary, vars_to_plot, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/ipw_logit_muster_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_muster, vars_to_plot, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/ipw_logit_seats_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_seats, vars_to_plot, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/ipw_cox_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wsurv, vars_to_plot, extract_coefs_coxph),
           pretty_dict, x_label = "Coefficient (Log Hazard Ratio)"),
         width = 10, height = 6, dpi = 300)
}
cat("Section 1 complete — 6 specs.\n")


===== IPW spec [total_raw] — treatment llandOwned =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_cbps` for information on how these are handled."



Call:
svyglm(formula = as.formula(paste("primary ~", rhs)), design = design, 
    family = quasibinomial(), data = rdf, weights = wts)

Survey design:
svydesign(~1, weights = wts, data = rdf)

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  -5.7193     1.7054  -3.354 0.000819 ***
llandOwned    0.8672     0.4662   1.860 0.063082 .  
ltitheOutT   -0.5727     0.3396  -1.686 0.092013 .  
lalmsInTot    0.2147     0.1179   1.821 0.068864 .  
smHouse      -1.3089     0.7222  -1.812 0.070172 .  
bigHouse      1.7639     0.7155   2.465 0.013810 *  
friary       -0.3086     0.9484  -0.325 0.744961    
mg_fsnub      0.1243     1.0427   0.119 0.905111    
mg_court     -0.2774     0.6287  -0.441 0.659102    
lLStax_pc     0.2290     0.5752   0.398 0.690669    
lpopC         0.4076     0.1889   2.158 0.031062 *  
distScot     -0.6370     0.5629  -1.132 0.257962    
area          0.2982     0.1417   2.105 0.035456 *  
uplands       0.3356     1.4281   0.235 0.814274  


===== IPW spec [total_sk] — treatment llo_sk =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_cbps` for information on how these are handled."


...


Call:
svyglm(formula = as.formula(paste("primary ~", rhs)), design = design, 
    family = quasibinomial(), data = rdf, weights = wts)

Survey design:
svydesign(~1, weights = wts, data = rdf)

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  -6.1745     1.4322  -4.311 1.74e-05 ***
lbg_sk        0.8939     0.3874   2.307 0.021191 *  
lsm_sk       -0.5642     0.3140  -1.797 0.072600 .  
lti_sk       -0.3998     0.3101  -1.289 0.197501    
lal_sk        0.1191     0.1433   0.831 0.406133    
smHouse      -1.1554     0.5267  -2.194 0.028425 *  
bigHouse      1.5903     0.5722   2.779 0.005520 ** 
friary        0.4554     1.0477   0.435 0.663847    
mg_fsnub     -0.2592     0.8969  -0.289 0.772643    
mg_court     -0.5256     0.5644  -0.931 0.351847    
lLStax_pc     0.2109     0.5499   0.383 0.701427    
lpopC         0.6390     0.1844   3.465 0.000547 ***
distScot     -0.4402     0.5656  -0.778 0.436479    
area          0.4999     0.1465   3.412 0.000665 *


===== IPW spec [split_arak] — treatment lbg_arak =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_cbps` for information on how these are handled."



Call:
svyglm(formula = as.formula(paste("primary ~", rhs)), design = design, 
    family = quasibinomial(), data = rdf, weights = wts)

Survey design:
svydesign(~1, weights = wts, data = rdf)

Coefficients:
            Estimate Std. Error t value Pr(>|t|)   
(Intercept)  -5.7364     1.7657  -3.249  0.00119 **
lbg_arak      0.4936     0.2666   1.851  0.06434 . 
lsm_arak     -0.2834     0.2543  -1.114  0.26532   
lti_arak     -0.3050     0.3864  -0.789  0.43008   
lal_arak      0.1585     0.1771   0.895  0.37094   
lni_arak      0.2155     0.2547   0.846  0.39748   
smHouse      -0.9871     0.6905  -1.430  0.15306   
bigHouse      1.3056     0.5877   2.221  0.02649 * 
friary       -0.4725     1.0419  -0.453  0.65028   
mg_fsnub      0.4765     1.0812   0.441  0.65950   
mg_court     -0.3376     0.6061  -0.557  0.57763   
lLStax_pc     0.1536     0.6796   0.226  0.82126   
lpopC         0.4165     0.1879   2.217  0.02677 * 
distScot     -0.3891     0.6756  -0.576  0.56473   
area        

Section 1 complete — 6 specs.


## Section 2: OwnOther specification (on-site vs off-site land per arable km²)

In [5]:
oo_specs <- list(
  ownOther_raw = list(
    treat       = "lotherLand",
    own_var     = "lownLand",
    other_covars = c("ltitheOutT", "lalmsInTot"),
    suffix      = "_ownOther_raw"
  ),
  ownOther_sk = list(
    treat       = "loth_sk",
    own_var     = "lown_sk",
    other_covars = c("lti_sk", "lal_sk"),
    suffix      = "_ownOther_sk"
  ),
  ownOther_arak = list(
    treat       = "loth_arak",
    own_var     = "lown_arak",
    other_covars = c("lti_arak", "lal_arak"), # Removed lni_arak to avoid collinearity
    suffix      = "_ownOther_arak"
  )
)

oo_results <- list()

for (spec_name in names(oo_specs)) {
  spec         <- oo_specs[[spec_name]]
  sfx          <- spec$suffix
  oo_treat     <- spec$treat
  oo_own       <- spec$own_var
  other_covars <- spec$other_covars

  oo_covars <- c(oo_own, other_covars, main_shared_after, covar_rhs)
  oo_balance_covars <- setdiff(oo_covars, oo_own)
  oo_all          <- c(oo_treat, oo_covars)
  oo_formula_rhs  <- paste(oo_all, collapse = " + ")

  cat(sprintf("\n===== OwnOther spec [%s] — treatment %s =====\n", spec_name, oo_treat))

  wt_oo       <- weightit(
    as.formula(paste(oo_treat, "~", paste(oo_balance_covars, collapse = " + "))),
    data = rdf, method = "cbps", over = FALSE
  )
  weights_oo  <- wt_oo$weights
  design_oo   <- svydesign(~1, weights = weights_oo, data = rdf)

  wlm_primary_oo <- svyglm(as.formula(paste("primary ~", oo_formula_rhs)),
                            data = rdf, weights = weights_oo,
                            design = design_oo, family = quasibinomial())
  wlm_muster_oo  <- svyglm(as.formula(paste("muster ~",  oo_formula_rhs)),
                            data = rdf, weights = weights_oo,
                            design = design_oo, family = quasibinomial())
  wlm_seats_oo   <- svyglm(as.formula(paste("seats ~",   oo_formula_rhs)),
                            data = rdf, weights = weights_oo,
                            design = design_oo, family = quasibinomial())
  wsurv_oo <- coxph(as.formula(paste("Surv(primary_survival, primary) ~", oo_formula_rhs)),
                    data = rdf, weights = weights_oo, robust = TRUE)

  oo_results[[spec_name]] <- list(
    muster = wlm_muster_oo, primary = wlm_primary_oo,
    seats  = wlm_seats_oo,  surv    = wsurv_oo,
    wts = weights_oo, treat = oo_treat, own_var = oo_own
  )

  # Back-compat aliases
  if (spec_name == "ownOther_arak") {
    wlm_muster_oo_main  <<- wlm_muster_oo
    wlm_primary_oo_main <<- wlm_primary_oo
    wlm_seats_oo_main   <<- wlm_seats_oo
    weights_oo_main     <<- weights_oo
  }

  oo_label_order <- c(oo_treat, oo_own, other_covars, "smHouse", "bigHouse",
                      "friary", "mg_fsnub", "mg_court",
                      "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  oo_cov_labels <- unlist(pretty_dict[oo_label_order])

  # Conley SEs (100km Bartlett) for the three svyglm logit columns
  se_muster_oo  <- weighted_conley_se(wlm_muster_oo,  weights_oo)
  se_primary_oo <- weighted_conley_se(wlm_primary_oo, weights_oo)
  se_seats_oo   <- weighted_conley_se(wlm_seats_oo,   weights_oo)

  tryCatch({
    stargazer(wlm_muster_oo, wlm_primary_oo, wlm_seats_oo, wsurv_oo,
      type = "latex",
      title = paste0("On-site vs. Off-site Monastic Land — IPW / Cox PH [", spec_name, "]"),
      label = paste0("tab:ipw_ownoff", sfx),
      column.labels = c("Muster", "Primary", "Seats", "Cox PH"),
      se = list(se_muster_oo, se_primary_oo, se_seats_oo, NULL),
      order = paste0("^", oo_label_order, "$"),
      covariate.labels = oo_cov_labels,
      omit = c("Constant", "uplands", "lowlands", "area", "mean_slope"),
      add.lines = list(
        c("Geographic Controls",  "Y", "Y", "Y", "Y"),
        c("CBPS weights",         "Y", "Y", "Y", "Y"),
        c("Conley SEs (100 km)",  "Y", "Y", "Y", "—"),
        c("Robust SEs (Lin-Wei)", "—", "—", "—", "Y"),
        c("CBPS treatment", rep(oo_treat, 4))
      ),
      align = TRUE,
      column.sep.width = ".5pt",
      omit.stat = c("aic", "lr", "wald", "logrank"),
      table.placement = "H",
      out = paste0("Output/Tables/IPW", sfx, ".tex")
    )
  }, error = function(e) {
    cat("Stargazer failed for spec", spec_name, ":", e$message, "\n")
  })

  vars_to_plot_oo <- c(oo_treat, oo_own, other_covars,
                       "smHouse", "bigHouse", "friary", "mg_fsnub", "mg_court",
                       "lLStax_pc", "wet_1535", "wet_1536", "lpopC")

  ggsave(paste0("Output/Images/Graphs/ipw_logit_primary_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_primary_oo, vars_to_plot_oo, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/ipw_logit_muster_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_muster_oo, vars_to_plot_oo, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/ipw_logit_seats_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wlm_seats_oo, vars_to_plot_oo, extract_coefs_svyglm),
           pretty_dict),
         width = 10, height = 6, dpi = 300)
  ggsave(paste0("Output/Images/Graphs/ipw_cox_coefficients", sfx, ".png"),
         plot = make_ipw_plot(
           make_coef_df_ipw(wsurv_oo, vars_to_plot_oo, extract_coefs_coxph),
           pretty_dict, x_label = "Coefficient (Log Hazard Ratio)"),
         width = 10, height = 6, dpi = 300)
}
cat("Section 2 complete — 3 ownOther specs.\n")


===== OwnOther spec [ownOther_raw] — treatment lotherLand =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_cbps` for information on how these are handled."



% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:11:23
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{On-site vs. Off-site Monastic Land — IPW / Cox PH [ownOther_raw]} 
  \label{tab:ipw_ownoff_ownOther_raw} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{1}{c}{paste("muster \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("primary \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("seats \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{\textit{survey-weighted}} & \multicolumn{1}{c}{\textit{survey-weighted}} & \multicolumn{1}{c}{\textit{survey-weighted}} & \mult


===== OwnOther spec [ownOther_sk] — treatment loth_sk =====


Warning message:
"Missing values are present in the covariates. See
`?WeightIt::method_cbps` for information on how these are handled."



% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:11:27
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{On-site vs. Off-site Monastic Land — IPW / Cox PH [ownOther_sk]} 
  \label{tab:ipw_ownoff_ownOther_sk} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{1}{c}{paste("muster \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("primary \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("seats \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{\textit{survey-weighted}} & \multicolumn{1}{c}{\textit{survey-weighted}} & \multicolumn{1}{c}{\textit{survey-weighted}} & \multic


===== OwnOther spec [ownOther_arak] — treatment loth_arak =====

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:11:31
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{On-site vs. Off-site Monastic Land — IPW / Cox PH [ownOther_arak]} 
  \label{tab:ipw_ownoff_ownOther_arak} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{1}{c}{paste("muster \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("primary \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("seats \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{\textit{survey-weighted}} & \multicolumn{1}{c}{\textit{surve

Section 2 complete — 3 ownOther specs.


## Section 3: Conley (Spatial HAC) Standard Errors — IPW Robustness

Spatial-HAC standard errors for the IPW (svyglm) outcome models. Because
`conleyreg` does not accept observation weights, we implement the weighted
sandwich directly: for each model we compute the IPW-weighted residuals
\(r_i = w_i (y_i - \mu_i)\), the working-weight bread \(B = (X' \mathrm{diag}(w \mu (1-\mu)) X)^{-1}\),
and the spatial meat \(M = X' \mathrm{diag}(r) \, \Omega \, \mathrm{diag}(r) X\)
where \(\Omega_{ij} = \max(1 - d_{ij}/c, 0)\) (Bartlett kernel) and \(d_{ij}\) is
the centroid-to-centroid km distance in EPSG:27700.
SEs reported at cutoffs of 50, 100, and 150 km, alongside the original
svyglm design-based SEs for comparison.

In [6]:
# --- Parish centroid coordinates (BNG meters) ----------------------------
parish_xy <- sf::st_coordinates(sf::st_centroid(sf::st_geometry(pdf)))
rdf$.cx <- parish_xy[, 1]
rdf$.cy <- parish_xy[, 2]

# --- Weighted spatial-HAC sandwich for an svyglm logit -------------------
# Returns a named numeric vector of SEs aligned to coef(svy_mod).
weighted_conley_se <- function(svy_mod, wts_full, cutoff_km, kernel = "bartlett") {
  mf <- model.frame(svy_mod)
  X  <- model.matrix(svy_mod)
  y  <- model.response(mf)
  mu <- as.numeric(fitted(svy_mod))
  used_rows <- as.integer(rownames(mf))
  w  <- wts_full[used_rows]
  cx <- rdf$.cx[used_rows]; cy <- rdf$.cy[used_rows]

  fam <- svy_mod$family$family
  if (grepl("binomial", fam)) {
    Avar <- mu * (1 - mu)
  } else if (grepl("poisson", fam)) {
    Avar <- mu
  } else {
    Avar <- rep(1, length(mu))
  }

  WX_bread <- X * (w * Avar)
  bread    <- solve(crossprod(X, WX_bread))
  u  <- as.numeric(w * (y - mu))
  Xu <- X * u

  d_km <- as.matrix(stats::dist(cbind(cx, cy))) / 1000
  K    <- pmax(1 - d_km / cutoff_km, 0)   # Bartlett

  meat <- crossprod(Xu, K %*% Xu)
  V    <- bread %*% meat %*% bread
  setNames(sqrt(diag(V)), colnames(X))
}

ipw_conley_cutoffs <- c(50, 100, 150)

# --- Helper: write one Conley-SE table (muster / primary / seats) --------
write_ipw_conley_table <- function(m_muster, m_primary, m_seats, wts,
                                   cutoff_km, title, label_suffix, out_path,
                                   covariate_labels, cov_order) {
  se_muster  <- weighted_conley_se(m_muster,  wts, cutoff_km)
  se_primary <- weighted_conley_se(m_primary, wts, cutoff_km)
  se_seats   <- weighted_conley_se(m_seats,   wts, cutoff_km)
  stargazer(
    m_muster, m_primary, m_seats,
    type             = "latex",
    se               = list(se_muster, se_primary, se_seats),
    title            = title,
    label            = paste0("tab:conley_ipw", label_suffix, "_", cutoff_km, "km"),
    column.labels    = c("Muster", "Primary", "Seats"),
    covariate.labels = covariate_labels,
    order            = paste0("^", cov_order, "$"),
    omit             = c("Constant", "uplands", "lowlands", "area", "mean_slope"),
    omit.stat        = c("aic", "lr", "wald", "logrank"),
    add.lines        = list(
      c("Conley cutoff (km)", rep(as.character(cutoff_km), 3)),
      c("Kernel",             rep("Bartlett",             3)),
      c("CBPS weights",       rep("Y",                    3))
    ),
    align            = TRUE,
    column.sep.width = ".5pt",
    table.placement  = "H",
    out              = out_path
  )
}

# --- (a) Main specs (6 total) --------------------------------------------
for (spec_name in names(main_results)) {
  res  <- main_results[[spec_name]]
  sfx  <- main_specs[[spec_name]]$suffix
  tr   <- res$tr
  monc <- res$monc

  cov_order  <- c(tr, monc, "smHouse", "bigHouse", "friary",
                  "mg_fsnub", "mg_court",
                  "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  cov_labels <- unlist(pretty_dict[cov_order])

  for (k in ipw_conley_cutoffs) {
    write_ipw_conley_table(
      m_muster         = res$muster,
      m_primary        = res$primary,
      m_seats          = res$seats,
      wts              = res$wts,
      cutoff_km        = k,
      title            = paste0("IPW Logits — Conley SEs [", spec_name, "], cutoff = ", k, " km"),
      label_suffix     = sfx,
      out_path         = paste0("Output/Tables/conley_ipw", sfx, "_", k, "km.tex"),
      covariate_labels = cov_labels,
      cov_order        = cov_order
    )
  }
  cat("Wrote Conley IPW tables for spec:", spec_name, "\n")
}

# --- (b) OwnOther specs (3 total) ----------------------------------------
for (spec_name in names(oo_results)) {
  res     <- oo_results[[spec_name]]
  sfx     <- oo_specs[[spec_name]]$suffix
  oo_tr   <- res$treat
  oo_own  <- res$own_var
  oo_oths <- oo_specs[[spec_name]]$other_covars

  cov_order  <- c(oo_tr, oo_own, oo_oths, "smHouse", "bigHouse", "friary",
                  "mg_fsnub", "mg_court",
                  "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  cov_labels <- unlist(pretty_dict[cov_order])

  for (k in ipw_conley_cutoffs) {
    write_ipw_conley_table(
      m_muster         = res$muster,
      m_primary        = res$primary,
      m_seats          = res$seats,
      wts              = res$wts,
      cutoff_km        = k,
      title            = paste0("IPW OwnOther — Conley SEs [", spec_name, "], cutoff = ", k, " km"),
      label_suffix     = sfx,
      out_path         = paste0("Output/Tables/conley_ipw", sfx, "_", k, "km.tex"),
      covariate_labels = cov_labels,
      cov_order        = cov_order
    )
  }
  cat("Wrote Conley IPW tables for spec:", spec_name, "\n")
}

cat("\nIPW Conley SE tables written to Output/Tables/conley_ipw_*.tex\n")



% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:11:34
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{IPW Logits — Conley SEs [total_raw], cutoff = 50 km} 
  \label{tab:conley_ipw_total_raw_50km} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{paste("muster \textasciitilde", rhs)} & \multicolumn{1}{c}{paste("primary \textasciitilde", rhs)} & \multicolumn{1}{c}{paste("seats \textasciitilde", rhs)} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\hline \\[-1.8ex] 
 ln(Total Monastic Land Owned + 1) & 0.379^{*} & 0.867^{*} & -0.111 \\ 
  & (0.223) & (0.50


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:11:52
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{IPW OwnOther — Conley SEs [ownOther_raw], cutoff = 50 km} 
  \label{tab:conley_ipw_ownOther_raw_50km} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{paste("muster \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("primary \textasciitilde", oo\_formula\_rhs)} & \multicolumn{1}{c}{paste("seats \textasciitilde", oo\_formula\_rhs)} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\hline \\[-1.8ex] 
 ln(Off-site Monastic Land + 1) & 0.422^{*} 


IPW Conley SE tables written to Output/Tables/conley_ipw_*.tex


## Section 4: Moran's I — Spatial Autocorrelation Diagnostics

Tests for residual spatial dependence in the IPW outcome models. Reports
Moran's I on Pearson residuals from the main (`per_km`) and `ownOther`
svyglm logits. Two spatial-weights matrices: row-standardized **queen
contiguity** and **k-nearest neighbors** (k=8). svyglm drops NA rows, so
the listw is subset to model-frame rows for each test.

In [7]:
# Moran's I on Pearson residuals from the IPW svyglm logits.
pacman::p_load(spdep, xtable)

# --- Build spatial weights once on the full parish set -------------------
nb_queen_ipw <- spdep::poly2nb(pdf, queen = TRUE)
lw_queen_ipw <- spdep::nb2listw(nb_queen_ipw, style = "W", zero.policy = TRUE)

parish_xy_ipw <- sf::st_coordinates(sf::st_centroid(sf::st_geometry(pdf)))
nb_knn8_ipw   <- spdep::knn2nb(spdep::knearneigh(parish_xy_ipw, k = 8))
lw_knn8_ipw   <- spdep::nb2listw(nb_knn8_ipw, style = "W")

# --- Helper: Moran's I on residuals from an svyglm, aligning W to used rows
morans_row_svy <- function(label, svy_mod, lw, lw_label) {
  used <- as.integer(rownames(model.frame(svy_mod)))
  r    <- residuals(svy_mod, type = "pearson")
  if (length(used) < length(lw$neighbours)) {
    lw_use <- spdep::subset.listw(
      lw, subset = seq_along(lw$neighbours) %in% used, zero.policy = TRUE
    )
  } else {
    lw_use <- lw
  }
  mt <- spdep::moran.test(r, lw_use, zero.policy = TRUE, na.action = na.omit)
  data.frame(
    Model    = label,
    Weights  = lw_label,
    n        = length(r),
    Morans_I = unname(mt$estimate["Moran I statistic"]),
    Expected = unname(mt$estimate["Expectation"]),
    Variance = unname(mt$estimate["Variance"]),
    z        = unname(mt$statistic),
    p_value  = unname(mt$p.value)
  )
}

# --- Models to test ------------------------------------------------------
ipw_resid_inputs <- list(
  "Main per_km — Muster"   = wlm_muster_main,
  "Main per_km — Primary"  = wlm_primary_main,
  "Main per_km — Seats"    = wlm_seats_main,
  "OwnOther — Muster"      = wlm_muster_oo,
  "OwnOther — Primary"     = wlm_primary_oo,
  "OwnOther — Seats"       = wlm_seats_oo
)

rows_ipw <- list()
for (lw_lab in c("Queen", "KNN-8")) {
  lw_i <- if (lw_lab == "Queen") lw_queen_ipw else lw_knn8_ipw
  for (nm in names(ipw_resid_inputs)) {
    rows_ipw[[length(rows_ipw) + 1L]] <- morans_row_svy(
      nm, ipw_resid_inputs[[nm]], lw_i, lw_lab
    )
  }
}
moran_tab_nb02 <- do.call(rbind, rows_ipw)
print(moran_tab_nb02, row.names = FALSE)

# --- Write LaTeX ---------------------------------------------------------
moran_tab_fmt2 <- moran_tab_nb02
moran_tab_fmt2$Morans_I <- sprintf("%.3f", moran_tab_fmt2$Morans_I)
moran_tab_fmt2$Expected <- sprintf("%.4f", moran_tab_fmt2$Expected)
moran_tab_fmt2$Variance <- sprintf("%.5f", moran_tab_fmt2$Variance)
moran_tab_fmt2$z        <- sprintf("%.2f", moran_tab_fmt2$z)
moran_tab_fmt2$p_value  <- ifelse(moran_tab_fmt2$p_value < 1e-4, "<1e-4",
                                  sprintf("%.4f", moran_tab_fmt2$p_value))
colnames(moran_tab_fmt2) <- c("Model", "Weights", "n", "Moran's I",
                              "E[I]", "Var(I)", "z", "p")

print(
  xtable(moran_tab_fmt2,
         caption = "Moran's I on Pearson residuals from IPW svyglm logits (nb02).",
         label   = "tab:moran_nb02",
         align   = c("l", "l", "l", "r", "r", "r", "r", "r", "r")),
  include.rownames  = FALSE,
  caption.placement = "top",
  table.placement   = "H",
  file = "Output/Tables/morans_nb02.tex"
)
cat("Wrote Output/Tables/morans_nb02.tex\n")


                 Model Weights    n     Morans_I      Expected     Variance
  Main per_km — Muster   Queen 1391  0.011671671 -0.0007215007 0.0002588004
 Main per_km — Primary   Queen 1391 -0.007311830 -0.0007215007 0.0002243000
   Main per_km — Seats   Queen 1391  0.087621695 -0.0007215007 0.0002668255
     OwnOther — Muster   Queen 1391  0.011078495 -0.0007215007 0.0002498016
    OwnOther — Primary   Queen 1391 -0.002960493 -0.0007215007 0.0001583594
      OwnOther — Seats   Queen 1391  0.105494869 -0.0007215007 0.0002690193
  Main per_km — Muster   KNN-8 1391  0.023293992 -0.0007215007 0.0001782354
 Main per_km — Primary   KNN-8 1391  0.039331866 -0.0007215007 0.0001544686
   Main per_km — Seats   KNN-8 1391  0.106250764 -0.0007215007 0.0001837637
     OwnOther — Muster   KNN-8 1391  0.025892807 -0.0007215007 0.0001720362
    OwnOther — Primary   KNN-8 1391  0.025948527 -0.0007215007 0.0001090432
      OwnOther — Seats   KNN-8 1391  0.116865371 -0.0007215007 0.0001852750
          z 

Wrote Output/Tables/morans_nb02.tex
